<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/5_Aprendizaje_supervisado/2_Taller_Regresion_Polinomica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# **Taller: Regresión Polinómica, Subajuste, Sobreajuste y Regularización**

**IMPORTANTE**: Guarda una copia de este notebook en tu Google Drive o computador.

**Taller en grupos de 3**

**Nombres estudiantes:**

- Juan Barrantes
- Miguel Rodriguez
- Juan Ordoñez

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma: “Taller_Reg_Pol_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/AUmhqMjhUK.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Plazo de entrega:**

30 de abril de 2026, máximo a las 11:59 p.m. Tenga en cuenta que luego de esa hora el formulario en forms se cierra. El Jupupyter Notebook también debe quedar subido en Github antes de esa hora.

**Instrucciones Generales:**

Completa el código en las celdas marcadas con `### TU CÓDIGO AQUÍ ###`. Puedes añadir más celdas si lo requieres.

## **Situación**

Una importante firma de inversión inmobiliaria te ha contratado como consultor de ciencia de datos. Su proceso actual de valoración de propiedades es lento y se basa en la intuición de unos pocos expertos. Quieren que desarrolles un modelo de machine learning para predecir el precio de venta de las viviendas (`SalePrice`) de forma más precisa y sistemática.

Te han entregado el dataset "Ames Housing", que contiene una gran cantidad de información sobre viviendas vendidas recientemente. Tu tarea es construir el mejor modelo posible, pero más importante aún, justificar por qué tu modelo es robusto y fiable, explicando cómo has manejado la complejidad y el riesgo de sobreajuste.

### **Ejercicio 1: Carga y Preparación Inicial de los Datos**

Primero, carguemos las librerías necesarias y el dataset. Nos enfocaremos en un subconjunto de las variables numéricas para mantener el taller manejable, pero el principio se aplica a todo el dataset.

In [1]:
### TU CÓDIGO AQUÍ ###
# Importación de librerías estándar
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Importación de herramientas de scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

Mejorar visualización de dataframes y gráficos

In [2]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

# Configuraciones para una mejor visualización
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

Cargar el dataset

In [3]:
# Usamos el dataset Ames Housing desde su fuente original (requiere internet)
url = 'http://jse.amstat.org/v19n3/decock/AmesHousing.txt'
df = pd.read_csv(url, sep='\t')
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.00,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,5,1960,1960,Hip,CompShg,BrkFace,Plywood,Stone,112.00,TA,TA,CBlock,TA,Gd,Gd,BLQ,639.00,Unf,0.00,441.00,"1,080.00",GasA,Fa,Y,SBrkr,1656,0,0,1656,1.00,0.00,1,0,3,1,TA,7,Typ,2,Gd,Attchd,"1,960.00",Fin,2.00,528.00,TA,TA,P,210,62,0,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.00,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.00,TA,TA,CBlock,TA,TA,No,Rec,468.00,LwQ,144.00,270.00,882.00,GasA,TA,Y,SBrkr,896,0,0,896,0.00,0.00,1,0,2,1,TA,5,Typ,0,NaN,Attchd,"1,961.00",Unf,1.00,730.00,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.00,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.00,TA,TA,CBlock,TA,TA,No,ALQ,923.00,Unf,0.00,406.00,"1,329.00",GasA,TA,Y,SBrkr,1329,0,0,1329,0.00,0.00,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,"1,958.00",Unf,1.00,312.00,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.00,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,7,5,1968,1968,Hip,CompShg,BrkFace,BrkFace,NaN,0.00,Gd,TA,CBlock,TA,TA,No,ALQ,"1,065.00",Unf,0.00,"1,045.00","2,110.00",GasA,Ex,Y,SBrkr,2110,0,0,2110,1.00,0.00,2,1,3,1,Ex,8,Typ,2,TA,Attchd,"1,968.00",Fin,2.00,522.00,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.00,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.00,TA,TA,PConc,Gd,TA,No,GLQ,791.00,Unf,0.00,137.00,928.00,GasA,Gd,Y,SBrkr,928,701,0,1629,0.00,0.00,2,1,3,1,TA,6,Typ,1,TA,Attchd,"1,997.00",Fin,2.00,482.00,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [4]:
# Para este taller, solo usaremos algunas columnas clave para simplificar.
df_sub = df[['Overall Qual', 'Gr Liv Area', 'SalePrice']].copy()
df_sub.columns = ['OverallQual', 'GrLivArea', 'SalePrice'] # Renombrar para facilidad
df_sub.head()

,OverallQual,GrLivArea,SalePrice
0,6,1656,215000
1,5,896,105000
2,6,1329,172000
3,7,2110,244000
4,5,1629,189900


**Explicación de las variables del dataset reducido**

1. SalePrice: Precio de Venta.

Esta es la variable objetivo. Es el precio final por el cual se vendió la propiedad, medido en dólares estadounidenses.

2. OverallQual: Calidad General.

Es una variable ordinal que califica la calidad general del material y el acabado de la casa. Es una de las variables predictoras (X) más importantes.

Escala: Va de 1 a 10.

10: Muy Excelente

9: Excelente

...

2: Pobre

1: Muy Pobre

3. GrLivArea: Área Habitable sobre el Nivel del Suelo.

Es una variable numérica que mide el total de metros cuadrados de área habitable que está por encima del nivel del suelo. No incluye el área del sótano. Es una de las variables predictoras (X) más fuertes, ya que, lógicamente, casas más grandes tienden a ser más caras.

In [5]:
df_sub.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   OverallQual  2930 non-null   int64
 1   GrLivArea    2930 non-null   int64
 2   SalePrice    2930 non-null   int64
dtypes: int64(3)
memory usage: 68.8 KB


### **Ejercicio 2: Dividir el conjunto de datos**

El método `.info()` muestra que no hay nulos en nuestro subconjunto. ¡Perfecto! Ahora, definamos nuestras variables `X` (predictoras) e `y` (objetivo) y dividamos los datos.

In [6]:
# Definir X e y
### TU CÓDIGO AQUÍ ###
x = df_sub.drop('SalePrice', axis=1)
y = df_sub['SalePrice']

In [7]:
# Dividir en entrenamiento y prueba (70/30)
### TU CÓDIGO AQUÍ ###
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [8]:
x_train.head()


,OverallQual,GrLivArea
2210,5,2002
782,5,1473
2310,6,1525
299,8,1191
2423,7,1414


In [9]:
y_train.head()

,SalePrice
2210,145000
782,143000
2310,183500
299,162500
2423,178740


### **Ejercicio 3: Modelo con Sobreajuste**

Vamos a crear un modelo muy complejo para ver qué tan mal puede generalizar.

**Crear un Modelo Polinómico de Grado 5**

Usa `Pipeline` para combinar `PolynomialFeatures` (grado 5), `StandardScaler` y `LinearRegression`. Entrénalo con los datos de entrenamiento.

In [10]:
# Crear el pipeline para el modelo polinómico
### TU CÓDIGO AQUÍ ###
pipeline_poly = Pipeline(steps=[
    ('poly_features', PolynomialFeatures(degree=5)),
    ('scaler', StandardScaler()),
    ('linear_regression', LinearRegression())
])

In [11]:
# Entrenar el modelo
### TU CÓDIGO AQUÍ ###
pipeline_poly.fit(x_train, y_train)

Pipeline(steps=[('poly_features', PolynomialFeatures(degree=5)),
                ('scaler', StandardScaler()),
                ('linear_regression', LinearRegression())])

In [12]:
# Calcular el error (RMSE) en entrenamiento y prueba y realizar un print de estos
### TU CÓDIGO AQUÍ ###

y_train_pred_poly = pipeline_poly.predict(x_train)
y_test_pred_poly = pipeline_poly.predict(x_test)

rmse_train_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_poly))
rmse_test_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_poly))

print(f'RMSE en entrenamiento (Polinómico grado 5): {rmse_train_poly:,.2f}')
print(f'RMSE en prueba (Polinómico grado 5): {rmse_test_poly:,.2f}')

RMSE en entrenamiento (Polinómico grado 5): 34,152.35
RMSE en prueba (Polinómico grado 5): 41,101.60


**Pregunta:** ¿Qué indica la diferencia que observas entre el error de entrenamiento y el de prueba? ¿Le recomendarías este modelo a la firma inmobiliaria? ¿Por qué?

Esto indica que el modelo está sobreajustado a los datos de entrenamiento. Ha aprendido los patrones y el ruido específico de los datos con los que fue entrenado, pero no logra generalizar bien a datos nuevos y no vistos.

No le recomendaría este modelo a la firma inmobiliaria. La razón es precisamente el sobreajuste. Un modelo sobreajustado no será fiable para predecir precios de propiedades nuevas

### **Ejercicio 4: Aplicar Regularización**

Ahora, vamos a "curar" el sobreajuste. Usaremos los mismos `PolynomialFeatures` de grado 5, pero cambiaremos el modelo de regresión.

**Implementar Regresión Ridge**

Copia el pipeline anterior, pero reemplaza `LinearRegression` con `Ridge(alpha=10)`. `alpha` es la fuerza de la penalización.

In [13]:
# Crear el pipeline para Ridge
### TU CÓDIGO AQUÍ ###
pipeline_poly2 = Pipeline(steps=[
    ('poly_features', PolynomialFeatures(degree=5)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=10))
])

In [14]:
# Entrenar el modelo
### TU CÓDIGO AQUÍ ###
pipeline_poly2.fit(x_train, y_train)

Pipeline(steps=[('poly_features', PolynomialFeatures(degree=5)),
                ('scaler', StandardScaler()), ('ridge', Ridge(alpha=10))])

In [16]:
# Calcular el error (RMSE) en entrenamiento y prueba y realizar un print de estos
### TU CÓDIGO AQUÍ ###
y_train_pred_poly = pipeline_poly2.predict(x_train)
y_test_pred_poly = pipeline_poly2.predict(x_test)

rmse_train_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_poly))
rmse_test_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_poly))

print(f'RMSE en entrenamiento (Ridge Polinómico grado 5): {rmse_train_poly:,.2f}')
print(f'RMSE en prueba (Ridge Polinómico grado 5): {rmse_test_poly:,.2f}')

RMSE en entrenamiento (Ridge Polinómico grado 5): 35,204.89
RMSE en prueba (Ridge Polinómico grado 5): 35,206.63


**Interpreta los resultados:**
La principal diferencia que observamos aquí, en comparación con el modelo Polinómico de Grado 5 sin regularización (donde el RMSE de entrenamiento era 34,152.35 y el de prueba 41,101.60), es que el RMSE de entrenamiento y el RMSE de prueba están ahora muy cerca el uno del otro.

Esto es una señal que indica que la Regresión Ridge ha logrado reducir significativamente el sobreajuste que presentaba el modelo polinómico simple. El modelo Ridge sigue siendo complejo (grado 5), pero la penalización alpha=10 ha evitado que se ajuste demasiado a los ruidos o particularidades de los datos de entrenamiento.

**Implementar Regresión Lasso**

Haz lo mismo, pero ahora con `Lasso(alpha=500, max_iter=10000)`. Lasso necesita un `alpha` más grande (porque la penalización L1 es diferente) y a veces más `max_iter` para converger.

In [23]:
# Crear el pipeline para Lasso
### TU CÓDIGO AQUÍ ###
pipeline_poly3 = Pipeline(steps=[
    ('poly_features', PolynomialFeatures(degree=5)),
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=500, max_iter=10000))
])

In [24]:
# Entrenar el modelo
### TU CÓDIGO AQUÍ ###
pipeline_poly3.fit(x_train, y_train)

Pipeline(steps=[('poly_features', PolynomialFeatures(degree=5)),
                ('scaler', StandardScaler()),
                ('lasso', Lasso(alpha=500, max_iter=10000))])

In [25]:
# Calcular el error (RMSE) en entrenamiento y prueba y realizar un print de estos
### TU CÓDIGO AQUÍ ###
y_train_pred_poly = pipeline_poly3.predict(x_train)
y_test_pred_poly = pipeline_poly3.predict(x_test)

rmse_train_poly = np.sqrt(mean_squared_error(y_train, y_train_pred_poly))
rmse_test_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_poly))

print(f'RMSE en entrenamiento (Polinómico grado 5): {rmse_train_poly:,.2f}')
print(f'RMSE en prueba (Polinómico grado 5): {rmse_test_poly:,.2f}')

RMSE en entrenamiento (Polinómico grado 5): 35,352.92
RMSE en prueba (Polinómico grado 5): 35,447.71


**Interpreta los resultados:**

Al igual que con la Regresión Ridge, observamos que el RMSE de entrenamiento y el RMSE de prueba están muy cerca el uno del otro. Esto es una señal positiva y similar a lo que vimos con Ridge, indicando que Lasso también ha logrado reducir el sobreajuste que presentaba el modelo polinómico simple sin regularización.

En comparación con el modelo Ridge (RMSE entrenamiento: 35,204.89, RMSE prueba: 35,206.63), el modelo Lasso presenta errores ligeramente superiores tanto en entrenamiento como en prueba. Esto puede deberse a la elección específica de alpha y a la naturaleza de la penalización L1 de Lasso, que tiende a llevar coeficientes a cero, realizando una selección de características más agresiva que Ridge.

**Selección de Variables con Lasso**

Una de las grandes ventajas de Lasso es que puede eliminar variables. Vamos a ver cuántas de las características polinómicas que creamos (ej. `OverallQual^2`, `GrLivArea^5`, `OverallQual * GrLivArea^4`, etc.) fueron eliminadas.

In [27]:
# Extraer los coeficientes del modelo Lasso
### TU CÓDIGO AQUÍ ###
coef_lasso = pipeline_poly3.named_steps['lasso'].coef_

In [28]:
# Extraer los nombres de las características generadas
# Guíate por el siguiente código: feature_names = pipeline_lasso.named_steps['poly_features'].get_feature_names_out(X.columns)
### TU CÓDIGO AQUÍ ###
feature_names = pipeline_poly3.named_steps['poly_features'].get_feature_names_out(x.columns)

In [29]:
# Contar cuántos coeficientes son exactamente cero
### TU CÓDIGO AQUÍ ###
num_zero_coefs = np.sum(coef_lasso == 0)

In [30]:
# Realizar un print de:
# Número total de características polinómicas generadas
# Número de características eliminadas por Lasso (coef = 0)
# Número de características CONSERVADAS
# Porcentaje de variables eliminadas
# Lista con los nombres de las características conservadas por Lasso (coef != 0)

### TU CÓDIGO AQUÍ ###
print(f'Número total de características polinómicas generadas: {len(feature_names)}')
print(f'Número de características eliminadas por Lasso (coef = 0): {num_zero_coefs}')
print(f'Número de características conservadas por Lasso (coef != 0): {len(feature_names) - num_zero_coefs}')
print(f'Porcentaje de variables eliminadas: {num_zero_coefs / len(feature_names) * 100:.2f}%')
print(f'Lista con los nombres de las características conservadas por Lasso (coef != 0): {feature_names[num_zero_coefs:]}')


Número total de características polinómicas generadas: 21
Número de características eliminadas por Lasso (coef = 0): 15
Número de características conservadas por Lasso (coef != 0): 6
Porcentaje de variables eliminadas: 71.43%
Lista con los nombres de las características conservadas por Lasso (coef != 0): ['OverallQual^5' 'OverallQual^4 GrLivArea' 'OverallQual^3 GrLivArea^2'
 'OverallQual^2 GrLivArea^3' 'OverallQual GrLivArea^4' 'GrLivArea^5']


**Interpreta los resultados:**

Inicialmente, creamos un modelo muy complejo con muchas combinaciones de nuestras variables originales (OverallQual y GrLivArea) hasta un grado 5. Esto nos dio 21 características posibles para el modelo.

Número de características eliminadas por Lasso (coef = 0): 15 y Porcentaje de variables eliminadas: 71.43%: Esto significa que el modelo Lasso, gracias a su penalización L1, identificó que 15 de esas 21 características eran poco relevantes para predecir el precio de la vivienda. Al establecer sus coeficientes en cero, Lasso las eliminó efectivamente del modelo. Esto es una forma de selección automática de características.

Número de características conservadas por Lasso (coef != 0): 6 y Lista con los nombres de las características conservadas: El modelo Lasso consideró que solo 6 de las 21 características generadas eran realmente importantes para la predicción. Estas son las características que tienen un peso (coeficiente) distinto de cero y, por lo tanto, contribuyen al modelo. Observamos que todas estas características conservadas son términos de grado 5, lo que sugiere que las interacciones y potencias más altas de OverallQual y GrLivArea son las más influyentes según este modelo.

### **Ejercicio 5: Conclusión y Recomendación para el Cliente**

**Resumir los Resultados**

Crea un `DataFrame` de pandas que compare el RMSE de entrenamiento y prueba de los tres modelos (Polinómico, Ridge, Lasso) y ordena los modelos según el RMSE de prueba de menor a mayor.

In [33]:
import pandas as pd
import numpy as np

# Crear DataFrame de resultados
# Los valores de RMSE se toman de los outputs de las celdas anteriores.
# Polinómico (Original): Train RMSE: 34,152.35, Test RMSE: 41,101.60 (de cYsRJl2mDT4S)
# Ridge Polinómico: Train RMSE: 35,204.89, Test RMSE: 35,206.63 (de ByyQb7LYD7zm)
# Lasso Polinómico: Train RMSE: 35,352.92, Test RMSE: 35,447.71 (de QMgyJ1ssENoI)
results_df = pd.DataFrame({
    'Modelo': ['Polinómico', 'Ridge', 'Lasso'],
    'RMSE_Entrenamiento': [34152.35, 35204.89, 35352.92],
    'RMSE_Prueba': [41101.60, 35206.63, 35447.71]
})

# Ordenar los modelos según el RMSE de prueba de menor a mayor
results_df = results_df.sort_values(by='RMSE_Prueba').reset_index(drop=True)

print(results_df)

       Modelo  RMSE_Entrenamiento  RMSE_Prueba
0       Ridge           35,204.89    35,206.63
1       Lasso           35,352.92    35,447.71
2  Polinómico           34,152.35    41,101.60


**Pregunta Final:**

Basado en tu análisis, ¿qué modelo le recomendarías a la firma inmobiliaria? Tu respuesta debe incluir:

1.  Una recomendación clara del mejor modelo.

Si la prioridad es la precisión predictiva, el modelo Ridge es el que le recomendaría a la firma inmobiliaria. Obtuvo el menor Error Cuadrático Medio (RMSE) en los datos de prueba (35,206.63), lo que significa que es el que mejor se generaliza a nuevas propiedades. Si la interpretabilidad del modelo es crucial, el modelo Lasso es una alternativa muy sólida.

2.  Una explicación en términos sencillos (para un gerente, no para un científico de datos) de por qué el modelo polinómico simple no era una buena idea, usando el concepto de "memorizar vs. generalizar".

Imaginen que el modelo polinómico simple es como un estudiante que estudia para un examen memorizando cada palabra de cada libro, incluyendo todos los errores tipográficos y las notas al margen. Cuando se le presenta una pregunta ligeramente diferente (una propiedad nueva), este estudiante se confunde porque no entendió los conceptos fundamentales, solo memorizó. Por eso, aunque parecía muy bueno en lo que ya conocía (los datos de entrenamiento), falló estrepitosamente en lo nuevo (los datos de prueba). No es fiable porque "memorizó" los datos pasados en lugar de "generalizar" y aprender cómo funciona realmente el mercado de bienes raíces.

3.  Una descripción de la ventaja principal del modelo Lasso en cuanto a la interpretabilidad y la selección de las características más importantes.

El modelo Lasso tiene una ventaja muy práctica: no solo predice, sino que también ayuda a entender qué es lo realmente importante. Piensen en él como un experto que revisa todos los posibles factores que afectan el precio de una casa (calidad, tamaño, interacciones complejas, etc.) y dice: "De estos 21 factores complicados, solo 6 son verdaderamente clave para determinar el precio". Al identificar y quedarse solo con los factores más influyentes (como las interacciones de alto grado entre la calidad general y el área habitable), Lasso nos da un modelo más sencillo, fácil de explicar y de entender. Esto es crucial porque no solo quieren un precio, sino también comprender qué impulsa ese precio para tomar mejores decisiones de negocio.